# 🎓 Machine Learning Pipeline Académique
## Système de Prédiction de Spécialisation pour Étudiants

**Projet:** Classification Multi-Classe (18 classes)

### 📋 Objectif
Construire un système ML pour prédire la spécialisation d'un étudiant.

### ⚠️ RÈGLE CRITIQUE
**Aucun modèle n'est entraîné avant que les données ne soient 100% nettoyées!**

Ordre strict: **Chargement → EDA → Nettoyage → Preprocessing → Entraînement**

In [ ]:
# Imports
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split, cross_val_score
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.linear_model import LogisticRegression
from sklearn.neighbors import KNeighborsClassifier
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.svm import SVC
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, confusion_matrix, classification_report
from sklearn.impute import KNNImputer
import warnings
warnings.filterwarnings('ignore')

%matplotlib inline
sns.set_style('whitegrid')
plt.rcParams['figure.figsize'] = (14, 8)

print('✅ Imports réussis')

# 1. Définition du Problème

- **Type:** Classification Multi-Classe (18 classes)
- **Target:** `specialization_label`
- **Features:** ~33 (25 questions + scores + niveaux + filière)

In [ ]:
# Chargement
df = pd.read_csv('dataset/student_tests.csv')
df_raw = df.copy()
print(f'Shape: {df.shape}')
display(df.head())

# 2. Description & Qualité du Dataset

Analyse AVANT nettoyage

In [ ]:
# Analyse qualité
print(f'Lignes: {len(df):,}')
print(f'Valeurs manquantes: {df.isnull().sum().sum():,}')
print(f'Doublons: {df.duplicated().sum():,}')

# Visualisation valeurs manquantes
missing = df.isnull().sum()[df.isnull().sum() > 0].sort_values(ascending=False)[:20]
plt.figure(figsize=(12, 8))
plt.barh(range(len(missing)), missing.values)
plt.yticks(range(len(missing)), missing.index)
plt.xlabel('Nombre de valeurs manquantes')
plt.title('Top 20 Colonnes avec Valeurs Manquantes')
plt.gca().invert_yaxis()
plt.tight_layout()
plt.show()

# 3. EDA (AVANT Nettoyage)

**⚠️ IMPORTANT:** EDA fait AVANT nettoyage pour comprendre l'état initial

In [ ]:
# EDA - Distributions
numeric_cols = df.select_dtypes(include=[np.number]).columns[:12]
fig, axes = plt.subplots(3, 4, figsize=(16, 12))
axes = axes.flatten()
for idx, col in enumerate(numeric_cols):
    axes[idx].hist(df[col].dropna(), bins=50, alpha=0.7)
    axes[idx].set_title(col)
plt.suptitle('Distributions des Features Numériques (AVANT NETTOYAGE)', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

# Distribution target
if 'specialization_label' in df.columns:
    target_dist = df['specialization_label'].value_counts()
    plt.figure(figsize=(14, 8))
    target_dist.plot(kind='bar')
    plt.title('Distribution de la Variable Cible', fontweight='bold')
    plt.xticks(rotation=45, ha='right')
    plt.tight_layout()
    plt.show()

# 4. Nettoyage des Données

Étapes: Doublons → Valeurs incohérentes → Valeurs manquantes → Outliers

In [ ]:
# 4.1 Doublons
print(f'Doublons avant: {df.duplicated().sum():,}')
df = df.drop_duplicates()
print(f'Doublons après: {df.duplicated().sum():,}')

# 4.2 Valeurs incohérentes
question_cols = [f'q{i}_score' for i in range(1, 26) if f'q{i}_score' in df.columns]
for col in question_cols:
    df[col] = df[col].clip(0, 3)

# Scores 0-100
for col in ['practical_test_score', 'logical_reasoning_score', 'problem_solving_score']:
    if col in df.columns:
        df[col] = df[col].clip(0, 100)

print('✅ Valeurs incohérentes corrigées')

In [ ]:
# 4.3 Valeurs manquantes
missing_before = df.isnull().sum().sum()
print(f'Valeurs manquantes avant: {missing_before:,}')

# Mean pour questions
for col in question_cols:
    if col in df.columns:
        df[col] = df[col].fillna(df[col].mean())

# KNN pour autres colonnes
numeric_cols_to_impute = ['practical_test_score', 'logical_reasoning_score', 'problem_solving_score',
                          'time_spent_minutes', 'previous_level', 'current_level', 'improvement_rate']
numeric_cols_to_impute = [col for col in numeric_cols_to_impute if col in df.columns]
if df[numeric_cols_to_impute].isnull().sum().sum() > 0:
    knn_imputer = KNNImputer(n_neighbors=5)
    df[numeric_cols_to_impute] = knn_imputer.fit_transform(df[numeric_cols_to_impute])

missing_after = df.isnull().sum().sum()
print(f'Valeurs manquantes après: {missing_after:,}')
print(f'✅ Supprimé: {missing_before - missing_after:,} valeurs manquantes')

In [ ]:
# 4.4 Outliers (IQR)
if 'time_spent_minutes' in df.columns:
    Q1 = df['time_spent_minutes'].quantile(0.25)
    Q3 = df['time_spent_minutes'].quantile(0.75)
    IQR = Q3 - Q1
    lower = Q1 - 1.5 * IQR
    upper = Q3 + 1.5 * IQR
    outliers = ((df['time_spent_minutes'] < lower) | (df['time_spent_minutes'] > upper)).sum()
    print(f'Outliers time_spent_minutes: {outliers:,}')
    df['time_spent_minutes'] = df['time_spent_minutes'].clip(lower, upper)
    print(f'✅ Outliers clippés')

df_cleaned = df.copy()
print(f'\n✅ Nettoyage terminé: {df_cleaned.shape}')

# 5. Preprocessing (APRÈS Nettoyage)

Encodage + Normalisation

In [ ]:
# Encodage
if 'filiere' in df_cleaned.columns:
    label_encoder = LabelEncoder()
    df_cleaned['filiere_encoded'] = label_encoder.fit_transform(df_cleaned['filiere'])

# Features
feature_cols = question_cols + numeric_cols_to_impute + ['filiere_encoded']
feature_cols = [col for col in feature_cols if col in df_cleaned.columns]
X = df_cleaned[feature_cols].copy()

# Target
y = df_cleaned['specialization_label'].copy()
target_encoder = LabelEncoder()
y_encoded = target_encoder.fit_transform(y)

print(f'Features: {len(feature_cols)}')
print(f'Classes: {len(target_encoder.classes_)}')

In [ ]:
# Split (stratifié)
X_train, X_test, y_train, y_test = train_test_split(
    X, y_encoded, test_size=0.2, random_state=42, stratify=y_encoded
)

print(f'Train: {X_train.shape}')
print(f'Test: {X_test.shape}')

# Normalisation
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

print('✅ Preprocessing terminé')

# 6. Entraînement des Modèles

5 modèles: Logistic Regression, KNN, Decision Tree, Random Forest, SVM

In [ ]:
# Modèles
models = {
    'Logistic Regression': LogisticRegression(max_iter=1000, random_state=42, n_jobs=-1),
    'KNN': KNeighborsClassifier(n_neighbors=5, n_jobs=-1),
    'Decision Tree': DecisionTreeClassifier(random_state=42, max_depth=20),
    'Random Forest': RandomForestClassifier(n_estimators=100, random_state=42, n_jobs=-1, max_depth=20),
    'SVM': SVC(kernel='rbf', random_state=42, probability=True)
}

results = {}
for name, model in models.items():
    needs_scaling = name in ['Logistic Regression', 'KNN', 'SVM']
    X_train_use = X_train_scaled if needs_scaling else X_train
    X_test_use = X_test_scaled if needs_scaling else X_test
    
    model.fit(X_train_use, y_train)
    y_pred = model.predict(X_test_use)
    
    results[name] = {
        'accuracy': accuracy_score(y_test, y_pred),
        'precision': precision_score(y_test, y_pred, average='weighted', zero_division=0),
        'recall': recall_score(y_test, y_pred, average='weighted', zero_division=0),
        'f1': f1_score(y_test, y_pred, average='weighted', zero_division=0),
        'y_pred': y_pred
    }
    print(f'{name}: Accuracy = {results[name]["accuracy"]:.4f}')

# Meilleur modèle
best_model_name = max(results.keys(), key=lambda x: results[x]['accuracy'])
print(f'\n🏆 Meilleur modèle: {best_model_name} ({results[best_model_name]["accuracy"]:.4f})')

# 7. Évaluation

In [ ]:
# Comparaison
comparison_df = pd.DataFrame({
    'Modèle': list(results.keys()),
    'Accuracy': [r['accuracy'] for r in results.values()],
    'Precision': [r['precision'] for r in results.values()],
    'Recall': [r['recall'] for r in results.values()],
    'F1-Score': [r['f1'] for r in results.values()]
}).sort_values('Accuracy', ascending=False)

display(comparison_df)

# Visualisation
plt.figure(figsize=(14, 8))
x = np.arange(len(comparison_df))
width = 0.2
plt.bar(x - 1.5*width, comparison_df['Accuracy'], width, label='Accuracy', alpha=0.7)
plt.bar(x - 0.5*width, comparison_df['Precision'], width, label='Precision', alpha=0.7)
plt.bar(x + 0.5*width, comparison_df['Recall'], width, label='Recall', alpha=0.7)
plt.bar(x + 1.5*width, comparison_df['F1-Score'], width, label='F1-Score', alpha=0.7)
plt.xlabel('Modèle')
plt.ylabel('Score')
plt.title('Comparaison des Modèles', fontweight='bold')
plt.xticks(x, comparison_df['Modèle'], rotation=45, ha='right')
plt.legend()
plt.grid(True, alpha=0.3, axis='y')
plt.tight_layout()
plt.show()

In [ ]:
# Matrice de confusion (meilleur modèle)
best_pred = results[best_model_name]['y_pred']
cm = confusion_matrix(y_test, best_pred)

plt.figure(figsize=(12, 10))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues')
plt.title(f'Matrice de Confusion - {best_model_name}', fontweight='bold')
plt.xlabel('Prédit')
plt.ylabel('Réel')
plt.tight_layout()
plt.show()

# Rapport de classification
print(f'\nRapport de Classification - {best_model_name}:\n')
print(classification_report(y_test, best_pred, target_names=target_encoder.classes_))

# 8. Conclusion & Interprétation

## Résultats

- **Meilleur modèle:** À déterminer selon résultats
- **Accuracy:** À déterminer selon résultats

## Impact du Nettoyage

Le nettoyage des données a été essentiel pour améliorer les performances.

## Limites

- Déséquilibre de classes
- Complexité du problème (18 classes)